In [1]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
import os 
from dotenv import load_dotenv
load_dotenv()

e:\RAG AGENTIC AI\RAG LEARNING\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
python-dotenv could not parse statement starting at line 7


True

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap


In [3]:
# step 1: Load and split the dataset

loader = TextLoader("langchain_crewai_dataset.txt")
row_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap =50)
chunks = splitter.split_documents(row_docs)

In [4]:
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [5]:
# step 2 : Vector store
embedding_model = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 453.40it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
vectorstore = FAISS.from_documents(chunks,embedding_model)

# step 3 : MMR Retriver
retriver = vectorstore.as_retriever(search_type ="mmr", search_kwargs={"k":5})


In [7]:
# step 4: LLM and Prompt

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [8]:
llm=init_chat_model(
    model="llama-3.1-8b-instant",
    model_provider="groq"
)

In [11]:
# Query expansion
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding rlevant tecnical terms,synoms , and helpful words

Original query: "{query}" 
                                                      
Expanded query:


""")
query_expansion_chain = query_expansion_prompt | llm | StrOutputParser()

In [12]:
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding rlevant tecnical terms,synoms , and helpful words\n\nOriginal query: "{query}" \n\nExpanded query:\n\n\n')
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000214F6C1E570>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000214F6D96BA0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))
| StrOutputParser()

In [13]:
query_expansion_chain.invoke({"query":"Langchain memory"})

'Here\'s the expanded query with relevant technical terms, synonyms, and helpful words:\n\n**Expanded Query:** \n\n* ("Langchain" OR "LLaMA" OR "Large Language Model" OR "LLM")\n* ("Memory" OR "Knowledge Graph" OR "Graph Database" OR "Semantic Memory")\n* (AND)\n* ("memory efficiency" OR "memory optimization" OR "knowledge graph management")\n* ("langchain architecture" OR "LLaMA implementation" OR "large language model design")\n\nThis expanded query includes:\n\n1. **Synonyms**:\n\t* "LLaMA" (LLaMA is a large language model developed by Meta AI)\n\t* "Large Language Model" (LLM is a type of AI model that can understand and generate human-like language)\n2. **Related technical terms**:\n\t* "Knowledge Graph" (a database that stores and manages knowledge and relationships)\n\t* "Graph Database" (a type of database that stores and manages graph-structured data)\n\t* "Semantic Memory" (a concept in artificial intelligence that refers to a memory system that can understand and reason abou

In [14]:
# RAG answering prompt
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below 
                                             
Context:
{context}
                                             
Question : {input}

""")

In [15]:
document_chain = create_stuff_documents_chain(llm=llm,prompt=answer_prompt)


In [16]:
# Step 5: Full RAG pipeline with query expansion
rag_pipeline = (
    RunnableMap({
        "input": lambda x: x["input"],
        "context": lambda x: retriver.invoke(query_expansion_chain.invoke({"query":x["input"]}))
    })
    | document_chain
)

In [17]:
#  Step 6: Run query
query = {"input": "What types of memory does LangChain support?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("Answer : \n",response)

To improve document retrieval, I will expand the original query by including relevant technical terms, synonyms, and helpful words. Here's the expanded query:

```json
{
  'input': [
    "What types of memory does LangChain support?",
    "LangChain memory types",
    "LangChain data storage options",
    "LangChain memory architectures",
    "Types of memory supported by LangChain",
    "LangChain memory capabilities",
    "Memory management in LangChain",
    "LangChain data persistence",
    "In-memory data storage with LangChain",
    "Types of data storage supported by LangChain",
    "Short-term memory in LangChain",
    "Long-term memory in LangChain",
    "Temporary memory in LangChain",
    "Persistent memory in LangChain",
    "Memory caching in LangChain",
    "Memory optimization in LangChain"
  ]
}
```

This expanded query includes a variety of related terms and phrases that can help improve document retrieval. Some of the key concepts and technologies included are:

* Typ

In [18]:
response = rag_pipeline.invoke(query)
print("Answer : \n",response)

Answer : 
 Based on the context provided, it does not directly mention the types of memory that LangChain supports. However, it does mention that LangChain integrates seamlessly with vector databases like FAISS, Chroma, Pinecone, and Weaviate. 

Vector databases are typically used for semantic search and are designed to handle large amounts of dense data, often referred to as embeddings.


In [19]:
#  Step 6: Run query
query = {"input": "What types of memory does crew ai support?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("Answer : \n",response)

To improve document retrieval, I'll expand the query by adding relevant technical terms, synonyms, and helpful words. Here's an expanded query:

Expanded Query: 
{'input': 'What types of memory architectures or storage systems does Crew AI support, specifically regarding volatile memory (e.g., RAM), non-volatile memory (e.g., flash, SSD), cache memory, and persistent memory (e.g., NVMe)?'}

Added technical terms and synonyms:

* 'memory architectures' and 'storage systems' to cover various memory types
* 'volatile memory' and 'non-volatile memory' to specify types of memory
* 'RAM' (Random Access Memory) as a synonym for volatile memory
* 'flash' and 'SSD' (Solid-State Drive) as synonyms for non-volatile memory
* 'cache memory' to cover a specific type of memory
* 'persistent memory' to specify a type of non-volatile memory
* 'NVMe' (Non-Volatile Memory Express) as a synonym for persistent memory

Helpful words:

* 'support' to specify that we're looking for information on compatibilit